<a href="https://colab.research.google.com/github/davidsmall-Durham/EDS-Practicals/blob/main/EDS_Practical_6_Part_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practical 6 - Part II: An introduction to Google Earth Engine

This practial introduces [Google Earth Engine (GEE)](https://earthengine.google.com/), a cloud-based computing platform designed for global-scale geospatial analysis and environmental monitoring. GEE combines a multi-petabyte dataset catalogue with Google's computing infrastructure, allowing large scale analyses of global datasets

In previous notebooks we downloaded data and wrote code to work on it (**data-to-code**). Working in GEE takes a fundamentally different approach; we build code and send it to Google's servers where it is executed on the data (**code-to-data**). Advantages include; we never download large amounts of raw data, and GEE can process data in parallel acorss its servers.

<br>

Three key concepts (remember the lecture) are:

1. **ImageCollections Vs. Images**. An `ee.Image` is a single 2D data array (or raster). An `ee.ImageCollection` is a stack of images making up a multi-dimensional datacube. Much like a NetCDF it is self describing with properties and a time dimension. Anything we create from an image (image collection) remains an image (image collection)

2. **GEE uses "Lazy Evaluation"**. It doesn't actually compute *anything* until we explictly tell it to. The vast majority of the code we use is simply stored as a set of instructions until we trigger computation with specific commands.

3. **Client-side Vs. Server-side**. The Client (laptop or browser) runs local code to assemble commands. The Server (Google’s cloud infrastructure) receives those instructions, parses them, and executes the processing when requested. Certain objects (e.g.,`ee.ImageCollection`) only exist on the server side and must be manipulated using server-side `.ee` methods.

<br>

A generalised GEE workflow often involves:
1. Select an `ee.ImageCollection` (e.g., ERA5 reanalysis)
2. **Filter** it by area, variable, time etc etc.
3. **Map** an operation or function over the filtered collection (e.g., Convert the units).
4.  **Reduce** the analysis to a single `ee.Image` (e.g., Calculate mean precipitation for the period of interest).
5. **Visualise or export** that image (or export multiple images e.g., to make a timeseries).

<br>

As an analogy, we write a recipe (code) to perform this workflow. The recipe includes ingredients (ImageCollection and filters) and instructions on how we want them cooked (what function/opetrations to carry out and how to reduce). We send the recipe to the kitchen (Google's servers) and when it is ready they export it back to us.

<br>

This notebook aims to....

1. Teach you how to select and filter an `ee.ImageCollection`.
2. Carry out simple analysis on the filtered collection.
3. Visualise results interactively using the `geemap` package.
4. Export results to cloud storage for future use.
5. Use online tools to debug Python code

**Firstly, copy the notebook to your Google Drive using the COPY TO DRIVE button.**



## **Initial setup**

As usual we first set up our Colab environment with the packages we need. Because we are going to work in Google Earth Engine we need to import the `ee` API that allows Python to interact with the GEE servers (this is already installed in Colab). We will also install two new packages:

*  `geemap`: A package that allows us to make interactive maps in Colab ([link](https://geemap.org/)).
*  `xee`: Xarray-Earth-Engine, an extension for `xarray` that is optimised to integrate with GEE ([link](https://xee.readthedocs.io/en/latest/)).

In [ ]:
#@title ##### Install and import packages
# Install geemap
!pip install geemap -q cartopy -q

# Install xarray and the official Google xee (Xarray-Earth-Engine) plugin (for export)
!pip install xee -q xarray -q netcdf4 -q rioxarray -q

# And import
import ee
import geemap
import xarray as xr
import pandas as pd
import xee
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import rioxarray




We need to authenticate with GEE in order to access its data sets and computing resources.

We intialise authentication with `ee.Authenticate()` and link to an existing GEE project. This needs to be created in the [EE code editor](https://code.earthengine.google.com/) before we start this process. The project ID can be found in the top right hand corner of the GEE code editor page. It never contains any spaces, (i.e. "your project" is rendered as "your-project").

In [ ]:
#@title #####Authenticate Google Earth Engine
# Trigger the authentication flow


# Initialize the Earth Engine module


## **Section 1: Set up our filtered Image Collection**

GEE hosts a vast amount of environmental (and remote sensing) data. You can search the entire collection in the [GEE Data Catalog](https://developers.google.com/earth-engine/datasets/catalog).

On this page search for "ERA5 daily". Click on the tab and it will open a page with more details. You will be able to see a "Snippet" that tells us that the ERA5 Dailly Image Collection is called `ee.ImageCollection("ECMWF/ERA5/DAILY")`. Scroll down and you will see a section headed "Explore with Earth Engine" with a tab "Colab(Python)". Under this tab there is some helpful code snippets including one that tells us how to select just the precipitation variable (called `total_precipitation`). Copy and paste that code snippet into the code cell below **but change the dates so we have 26 years of precipitation data spanning a perdiod from 1st Jan 1979 to 31st Dec 2004.**

**NOTE**: The ERA5 DAILY dataset on GEE only runs to July 2020. The HOURLY dataset is more up-to-date.

In [ ]:
#@title ##### **Input Needed**: Input code to set up filtered `ee.ImageCollection`
# Daily total precipitation sums


Sticking with our recipe analogy, this is us defining our **ingredients list**. We haven't downloaded any data (check the RAM counter) or triggered any processing on the GEE servers.

Go back to the [ERA5 Daily page](https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_DAILY) in GEE Data Catalog page and select the "Bands" tab underneath the image. Here it gives lots of useful information (metadata) for all of the variables (bands) in this dataset.

**Questions**
1.  What are the units of the precipitation variable?
2.  What is the spatial resolution (grid cell size) of the ERA5 data set?
3.  How many individual images are in our filtered image collection?

### **Section 1.1 Slice to a particualr AOI**

An Image Collection such as `era5_tp` is simialr to a NetCDF file. As a result many of the basic operations fall under the same categories of **slicing** and **aggregating**.

We already used the `.select` and `.filter` commands to slice the ERA5 image collection. Next, we can slice `era5_tp` to a specfic geographic area.

We will look at the area affected by the Asian Monsoon. We define our AOI using coordinates.  Because GEE runs operations on Google's servers, standard Python geometric shapes won't work. We must create a server-side object using the `ee.Geometry.Rectangle` command. Passing this geometry into `Filter.bounds()` tells GEE to ignore everything that doesn't intersect with our AOI. That data still exists but no further operations will be carried out on it.


In [ ]:
# Define an Area of Interest (AOI)
# Format: ee.Geometry.Rectangle([min_longitude, min_latitude, max_longitude, max_latitude])
# This box covers India, Southeast Asia, and parts of China/Indian Ocean


# Filter the collection spatially.


We have now defined our ingredients. Nothing has been processed but we have written a list of instructions telling GEE how to filter the raw ERA5 dataset to the particualr subsset of data we want to work with.

## **Section 2: Define the calculations to be undertaken**

We are going to keep things simple and compare average rainfall between a pre-monsoon month (April) and a peak monsoon month (July). To do this we will:

1. **Slice** out every daily image for those months across all 26 years.
2. **Transform (map)** individual images to convert units from metres to millimetres.
3. **Aggregate (Reduce)** those daily images into a single long-term average image for each month.

We use  `ee.Filter.calendarRange` to filter by month. Unlike `Filter.date`, which requires continuous start/end dates, `.calendarRange` is **cyclical**. By specifying the 'month' parameter (e.g., 4 for April), GEE selects every April daily image from 1979 through 2004, ignoring the specific year.

When working with an ImageCollection, we often perform operations in two ways:

*   **Mapping** (`.map()`) applies an operation to every individual image in a collection independently, returning a new ImageCollection. For example, ERA5 total precipitation is recorded in meters ($m$). To convert every daily image to millimeters ($mm$), we map a function that multiplies each image by $1000$.

*  **Aggregating** (`.sum()`): Combines all images in the collection down into a single ee.Image. Calling `.sum()` adds up every daily rainfall image across all 26 years into one single cumulative image.


Rather than writing out the same sequence of code twice for April and July, we wrap our slicing, mapping, and aggregating steps inside a simple Python function:

In [ ]:
#@title #### Define a simple function to calculate monthly mean




You will see in the above code that we `.clip` our `monthly_mean` to our AOI. There is a fundamental difference between filtering and clipping.

* **Filtering** is a Metadata operation. It reads the spatial bounding box on the file's metadata tag and asks: "Does this image intersect with the AOI?" If no, it ignores the whole file. If yes, it keeps the whole file. It has **low computational cost**

* **Clipping** is a Pixel operation. It inspects every pixel in the grid and physically masks out those lying outside your boundary geometry. Processing millions of individual pixels requires real computing power. It has **high computational cost**.

To keep scripts running smothly, a good workflow for GEE is:

1. Filter first. Rapidly narrow down a massive ImageCollection spatially (.`Filter.bounds()`) and temporally (e.g., `.Filter.date()`).
2. Do maths next: Perform all arithmetic, transformations (`.map()`), and aggregations (e.g., `.sum()`, `.mean()`) on the image stack.
3. Clip last: Apply `.clip()` to the single, aggregated output image right before you map, display, or export it.

**Questions**


1.  Why do we use `ee.Geometry.Rectangle()` to define our study area instead of creating a standard Python bounding box list like `[60, 0, 140, 45]`?
2. When we use `ee.Filter.calendarRange(4, 4, 'month')` on our ERA5 collection  (1979-2004), how many images are returned?
3. Why do we clip at the end of this sequence rather than at the beginning? - This may seem counter intuitive.
4. Does calling `.sum()` on an ImageCollection return a single `ee.Image` or a single number?



##**Section 3: Visualising our calcualted monthly averges**

We have now written instructions to calcualte two `ee.Image` objects (2D aggregations of April and July rainfall for our 26-year period).

Up to this point, no pixels have actually been processed on Google’s servers. We have simply stored our recipe on our client machine (Colab).

To tell GEE to execute this recipe, we can use the interactive map tools provided by the `geemap` library.

`Map.addLayer()' takes our `ee.Image` objects (`April_average` and `July_average`) and sends a request to the GEE servers. They execute our recipe and displays the results on our screen.

In [ ]:
#@title ##### Visualise April Vs July rainfall
# Initialize the map centered on India


# Add an interactive basemap selector tool


# High-contrast visualization parameters for monthly totals


# Add layers to the map


# Add the Colour Bar Legend


# And show



###**Questions**

1. In total, how many daily images did we extract from the global ERA5 Image Collection for our 26 year period?
2. If the resolution of a grid cell in the ERA5 dataset is ~$0.25^\circ$ x $0.25^\circ$, how many grid cells (or pixels) does our AOI contain?
3. From these two calculations, how many individual data points did we just process (sum, multiply and divide)?

## **Section 4: Exporting results**

The interactive maps produced by `geemap` are not 'publication' standard. You wouldn't include one in a report or dissertation. However, we can export our aggregate `ee.Image` objecys to our Google Drive and plot them more professionally using `cartopy`.

<br>

We don't need to mount our Google Drive to export the files. When we run `ee.batch.Export.image.toDrive()`, the data transfer happens server-to-server. Your Google Colab notebook is just sending a command. Within the above command we pass a numuber of variables that tell GEE what to output in our file.

<br>

Note: when we click 'play' in this code cell it execute's almost instantly -0 **this does not mean the export is complete**. The actual export happens in the background on the Google servers. To check if it has finished is to go to the [GEE code editor](https://code.earthengine.google.com/) and click on the 'Tasks' tab. This export request is pretty simple so it should finish very quickly (< 1 minute for both).

N.B. `start()` is another command that triggers computation on the GEE servers.

In [ ]:
#@title #### This code cell defines the export configurations


Once the export is complete we are nearly ready to plot the results.

However, while we didn't need to mount our drive to export the files *we do* need to mount our drive to plot the results in this notebook.

In [ ]:
#@title ##### This cell mounts your Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define the path to the folder where you saved your files


Navigate to the folder you exported the files to, using the folder panel on the left of Colab, you should now seem them in your Drive.

To read a GeoTIFF directly into an `xarray.DataArray`, Python uses an extension package called `rioxarray`.

We first define our paths so Python can 'see' the files in our Google Drive.

We then open the data arrays using `rioxarray`.

*Note*: When Earth Engine exports a GeoTIFF, it automatically structures the file to accommodate multi-band satellite imagery (e.g., red, green, blue, near-infrared bands). Because we only exported a single variable (precipitation), our file contains a "band" dimension with a length of exactly 1. `cartopy` can't plot 3D arrays (Shape: `[1, Latitude, Longitude]`) so we need to get rid of this 'band' dimension. To do this, we use the `.squeeze()` function. This command cuts any dimension of length 1 flattening our structure into a clean 2D grid (this only edits the meta-data, not the actual data).

We can now write some code to plot a figure. You have probably noticed that it takes quite a lot of code to plot a nice figure. Don't be intimidated by this, most of it is quite '*boilerplate*' and can easily be reused with only minor edits.

In [ ]:
#@title ##### This code cell loads the GeoTiffs into xarray and plots a nice figure

################################################################################

# define the paths and load

# define the full paths to the files (we could type them by hand but this is easier)


# Load Data directly into xarray. We need to use .squeeze as the geotiff has a
# dimensions of (1, lat, lon). .squeeze() removes dimensions of length one
# allowing cartopy to plot the flattened 2D array


################################################################################

# Now we write our plotting code.

# Set projection and set up 2 panel figure
projection = ccrs.PlateCarree()
fig, axes = plt.subplots(1, 2, figsize=(16, 7), subplot_kw={'projection': projection})

# Mapping configurations
v_min = 0
v_max = 600
cmap_choice = 'YlGnBu'
map_extent = [60, 140, 0, 45] # [Min Lon, Max Lon, Min Lat, Max Lat]

# Plot April (Left Panel)
ax1 = axes[0] # tells python what axes to plot into (remember python indexing starts at 0 so [0] means use the first set of axes)
ax1.set_extent(map_extent, crs=projection)
ax1.add_feature(cfeature.COASTLINE, linewidth=1.2, edgecolor='#2b2d42')
ax1.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor='#6c757d', linestyle=':')

# xarray handles geographic coordinates natively.
# We set add_colorbar=False because we will build a shared one at the bottom.
img1 = da_april.plot.imshow(
    ax=ax1,
    transform=projection,
    cmap=cmap_choice,
    vmin=v_min,
    vmax=v_max,
    add_colorbar=False
)

# Add and format gridlines for the first subplot
gl1 = ax1.gridlines(draw_labels=True, dms=False, x_inline=False, y_inline=False)
# turn off labels on top and rhs for clarity
gl1.top_labels = False
gl1.right_labels = False
ax1.set_title('April Average Rainfall 1979-2004 (Pre-Monsoon)', fontsize=12, fontweight='bold', pad=10)


# Plot July (Right Panel)
ax2 = axes[1] # tells python what axes to plot into (remember python indexing starts at 0 so [1] means use the second set of axes)
ax2.set_extent(map_extent, crs=projection)
ax2.add_feature(cfeature.COASTLINE, linewidth=1.2, edgecolor='#2b2d42')
ax2.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor='#6c757d', linestyle=':')

img2 = da_july.plot.imshow(
    ax=ax2,
    transform=projection,
    cmap=cmap_choice,
    vmin=v_min,
    vmax=v_max,
    add_colorbar=False
)

# Add and format gridlines for the second subplot
gl2 = ax2.gridlines(draw_labels=True, dms=False, x_inline=False, y_inline=False)
gl2.top_labels = False
gl2.left_labels = False
ax2.set_title('July Average Rainfall 1979-2004 (Peak Monsoon)', fontsize=12, fontweight='bold', pad=10)


# Add a Shared Colorbar
cbar_ax = fig.add_axes([0.25, 0.2, 0.5, 0.025]) # [left, bottom, width, height]
cbar = fig.colorbar(img2, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Total Monthly Precipitation (mm)', fontsize=11, fontweight='semibold', labelpad=5)

plt.subplots_adjust(wspace=0.15, bottom=0.15) # adjust spacing

# These code lines save the plot
# Define the output path and filename
output_save_path = '/content/drive/MyDrive/GEE_Monsoon/monsoon_comparison_map.png'
plt.savefig(output_save_path, dpi=300, bbox_inches='tight')

plt.show()

##**Summary and reflection**

Hopefully from your answers to the above questions you can see that despite our code and analysis being very simple it still involved processing a large amount of data.

It is worth reflecting on how we would might have appraoched this using the traditional "**data-to-code**" approach we adopted in the previous notebooks.
*  The Copernicus Climate Data Store (CDS) limits single-request sizes to manage server loads. You would have to manually write API loops to request and download 26 individual NetCDF files (one year at a time).
*  You would need to hard code local file paths, allocate signficant amounts of hard drive space, and ensure your local Python environment didn't crash while trying to read large arrays into your laptop's limited RAM.
*  If you decided to change your Area of Interest (AOI) or look at a different month (like September's monsoon retreat), your existing data would be useless. You would have to re-code the API request, rejoin the server queue, and re-download hundreds of megabytes all over again.

### **Questions: What are the advantages of Earth Engine?**


* Why was our local Colab RAM barely touched during this exercise?

* How many lines of code would you  need to modify if you wanted to apply this  analysis to different locations, months, or decades?


---
## **Section 5: Task — Comparing 2003 and 2019 European Heatwaves**

In the summer of 2003, Western Europe experienced an iconic, historic heatwave. In 2019, severe heatwaves again brought record-breaking summer temperatures across Europe.

Your task is to analyze ERA5 daily mean temperatures (`ECMWF/ERA5/DAILY`) over Western Europe to evaluate how 2019 compares both to a long-term climate baseline and directly to 2003.

### **Your Goal**
1. **Filter**: Select `ECMWF/ERA5/DAILY`and `mean_2m_air_temperature`.

2. **Filter to Western Europe**: Define an AOI.

3. **Use a Python function to compute 3 Summer Means (in Celsius)**:
   * *Baseline Mean*: 20-year climatology baseline (**1981-2000**).
   * *Summer 2003 Mean*: June, July, August, 2003.
   * *Summer 2019 Mean*: June, July, August 2019.

5. **Compute 2 Anomalies**:
   * *Long-Term Anomaly*: $\text{Summer 2019} - \text{Baseline}$
   * *Event-to-Event Difference*: $\text{Summer 2019} - \text{Summer 2003}$

6. **Interactive Map**: Visualize the **2019 Anomaly** and **2019 vs 2003 Difference** You can add multiple layers using `geemap`.

7. **Export & Cartopy Plot**: Export the difference GeoTIFF to Google Drive and plot a publication-ready figure using Cartopy with a diverging colormap (`RdBu_r` or `coolwarm`).

### **Code blocks**

The code blocks below contain code to complete the task set above. However there are numerous bugs in the code.

Your job is to de-bug the code.

The aim here is to help you understand
1.  How the GEE workflow can throw errors that are not immediately apparent.
2.  How certain operations cannot be performed on certain object types
3.  Emphasise the concept of lazy evaluation

In [ ]:
#@title ##### Input code to set up initial `ee.ImageCollection'

# Select the image collection and specific variable
era_all = (
    ee.ImageCollection('ECMWF/ERA5/DAILY')              # The name of the overarching image collection in the data catalog.
    .select('mean_2m_air_temperature')                  # select only the variable with this name
)

In [ ]:
# @title #####Define an Area of Interest (AOI)
# Format: ee.Geometry.Rectangle([min_longitude, min_latitude, max_longitude, max_latitude])
europe_aoi = ee.Geometry.Rectangle([]) #<-------------------Insert coordinates here

# Filter the collection spatially.
era_europe = er_all.filter(ee.Filter.bounds(europe_aoi))

In [ ]:
# @title #####Define and implement a python function to calculate the summer mean temperatures in degrees Celsius

def calculate_summer_means(collection, start_date, end_date, aoi):
    """Slices by date range, maps unit conversion (K -> °C) across daily images,

    calculates the temporal mean, and clips to the AOI.
    """

    #Filter by temporal range
    filtered = collection.filter(ee.Filter.date(start_date, end_date))

    # Check if it is one year or multi-year; filter for June, July, August (JJA)
    start_yr = ee.Date(start_date).get("year")  # what is year of our start_date
    end_yr = ee.Date(end_date).get("year")      # what is year of our end_date
    is_multi_yr = start_yr.neq(end_yr)          # If they are not the same then the collection spans multiple years [this returns a logical]

    # now we filter for JJA
    JJA_filtered = ee.ImageCollection(
        ee.Algorithms.If(
          is_multi_yr,
          filtered.filter(ee.Filter.calendarRange(6,8,"month")),
          filtered,
        )
      )

    # map the celsius conversion across all images
    filtered_celsius = JJA_filtered.map(lambda img: img.subtract(-273.15))

    # Reduce by taking the mean across time and clip to a single image for europer
    # This is what the function "RETURNS"
    return filtered_celsius.mean().clip(aoi)


# Now we can use our function to calculate the summer means for the periods we are interested in
summer_2019 = calculate_summer_means(era_europe, '2019-06-01', '2019-08-31', europe_aoi)
summer_2003 = calculate_summer_means(era_europe, '2003-06-01', '2003-08-31', europe_aoi)
baseline = calculate_summer_means(era_europe, '1981-06-01', '2000-08-31', europe_aoi)

In [ ]:
# @title ##### Calculate Temperature Anomalies (°C)

# Calculate the 2019 heatwave anomaly relative to the 2003 heatwave
anomaly_2019_vs_2003 = era_europe.subtract(summer_2003)

# Calculate the 2019 heatwave anomaly relative to the long-term baseline (1981–2000)
anomaly_2019_vs_baseline = era_europe.subtract(baseline)

In [ ]:
#@title #####Visualise in GEEMAP
Map_heatwaves = geemap.Map(center=[48, 5], zoom=4)

# Add the interactive basemap selector tool
Map_heatwaves.add("basemap_selector")

vis_anomaly = {
    'min': -3.0,
    'max': 3.0,
    'palette': ['#0571b0', '#92c5de', '#f7f7f7', '#f4a582', '#ca0020']
}

# Add layers
Map_heatwaves.addLayer(anomaly_2019_vs_baseline, vis_anomaly, '2019 Anomaly (vs 1981-2000)')
Map_heatwaves.addLayer(anomaly_2019_vs_2003, vis_anomaly, '2019 Anomaly (vs 2003)')
Map_heatwaves.add_colorbar(vis_params=vis_anomaly, label="Temperature Difference (°C)", orientation="horizontal")
Map_heatwaves

### **DO BOTH OF THE ANOMALY LAYERS LOOK SENSIBLE??**

If you have caught all the bugs they should look sensible!

In [ ]:
# Export anomalies

vBaseline_anomaly = ee.batch.Export.image.toDrive(
    image=anomaly_2019_vs_baseline,                        # The image layer you want to export
    description='Summer_temp_anomaly_2019_baseline',  # Name of the task in GEE (can be anything)
    folder='GEE_Heatwave',                            # Folder name that will be created in your Google Drive
    fileNamePrefix='Anomaly_2019-baseline',           # The actual name of the downloaded GeoTIFF file
    scale=27830,                                      # Resolution in meters
    region=era_europe,                                # Bounding geometry box to clip the data to
    fileFormat='GeoTIFF',                             # The spatial raster format
    crs='EPSG:4326'                                   # Coordinate reference system (WGS84 Lat/Lon)
)


v2003_anomaly = ee.batch.Export.image.toDrive(
    image=anomaly_2019_vs_2003,                            # The image layer you want to export
    description='Summer_temp_anomaly_2019-2003',      # Name of the task in GEE (can be anything)
    folder='GEE_Heatwave',                            # Folder name that will be created in your Google Drive
    fileNamePrefix='Anomaly_2019-2003',               # The actual name of the downloaded GeoTIFF file
    scale=27830,                                      # Resolution in meters
    region=era_europe,                                # Bounding geometry box to clip the data to
    fileFormat='GeoTIFF',                             # The spatial raster format
    crs='EPSG:4326'                                   # Coordinate reference system (WGS84 Lat/Lon)
)



###**REMEMBER TO CHECK ON THE CODE EDITOR TO SEE IF THE EXPORT TASKS HAVE COMPLETED**

GEE is lazy and doesn't process anything until we explictly ask it to!!

In [ ]:
#@title #### Plot the anomalies in using Cartopy

# define the directory files were exported to
DATADIR = "/content/drive/My Drive/GEE_Heatwave/"

# Define the full paths to the files
vBaseline_anomaly_path = os.path.join(DATADIR, 'Anomaly_2019-baseline.tif')
v2003_anomaly_path = os.path.join(DATADIR, 'Anomaly_2019-2003.tif')

# Load Data directly into xarray. Remember the geotiff has a
# dimensions of (1, lat, lon)!!
da_vBaseline = rioxarray.open_rasterio(vBaseline_anomaly_path).squeeze()
da_v2003 = rioxarray.open_rasterio(v2003_anomaly_path).squeeze()


# Set projection and set up 2 panel figure
projection = ccrs.PlateCarree()
fig, axes = plt.subplots(1, 2, figsize=(16, 7), subplot_kw={'projection': projection})

# Mapping configurations
v_min = 0
v_max = 600
cmap_choice = 'YlGnBu'
map_extent = []

# Plot baseline anomaly (Left Panel)
ax1 = axes[0] # tells python what axes to plot into (remember python indexing starts at 0 so [0] means use the first set of axes)
ax1.set_extent(map_extent, crs=projection)
ax1.add_feature(cfeature.COASTLINE, linewidth=1.2, edgecolor='#2b2d42')
ax1.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor='#6c757d', linestyle=':')

# We set add_colorbar=False because we will build a shared one at the bottom.
img1 = da_vBaseline.plot.imshow(
    ax=ax1,
    transform=projection,
    cmap=cmap_choice,
    vmin=v_min,
    vmax=v_max,
    add_colorbar=False
)

# Add and format gridlines for the first subplot
gl1 = ax1.gridlines(draw_labels=True, dms=False, x_inline=False, y_inline=False)
# turn off labels on top and rhs for clarity
gl1.top_labels = False
gl1.right_labels = False
ax1.set_title('April Average Rainfall 2000-2025 (Pre-Monsoon)', fontsize=12, fontweight='bold', pad=10)


# Plot 2003 anomaly (Right Panel)
ax2 = axes[1]
ax2.set_extent(map_extent, crs=projection)
ax2.add_feature(cfeature.COASTLINE, linewidth=1.2, edgecolor='#2b2d42')
ax2.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor='#6c757d', linestyle=':')

img2 = da_v2003.plot.imshow(
    ax=ax2,
    transform=projection,
    cmap=cmap_choice,
    vmin=v_min,
    vmax=v_max,
    add_colorbar=False
)

# Add and format gridlines for the second subplot
gl2 = ax2.gridlines(draw_labels=True, dms=False, x_inline=False, y_inline=False)
gl2.top_labels = False
gl2.left_labels = False
ax2.set_title('July Average Rainfall 2000-2025 (Peak Monsoon)', fontsize=12, fontweight='bold', pad=10)


# Add a Shared Colorbar
cbar_ax = fig.add_axes([0.25, 0.2, 0.5, 0.025]) # [left, bottom, width, height]
cbar = fig.colorbar(img2, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Total Monthly Precipitation (mm)', fontsize=11, fontweight='semibold', labelpad=5)

plt.subplots_adjust(wspace=0.15, bottom=0.15) # adjust spacing

# These code lines save the plot
# Define the output path and filename
output_save_path = '/content/drive/MyDrive/GEE_Heatwave/Anomaly_comparison_map.png'
plt.savefig(output_save_path, dpi=300, bbox_inches='tight')

plt.show()

## **Extra Activities**

If you finish with time to spare, or in your own time if you wish, try writing a little more code to tackle the tasks below.

1.   What was the average anomaly across the AOI (2019 vs. baseline)?
2.   What was the maximum anomal across the AOI (2019 vs. baseline)
3.   Finally, in the code above we seperate the calculation of the mean from the calculation of the anomaly. It would be quite strightforward to write a function that could do both by adding an additional (optional) parameter to specify a baseline_image to subtract if desired. Try to write a function that can do both calcualtions.

